In [3]:
from geonames import GeoNames
g = GeoNames()


  Using cached pandas-3.1.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (16 kB)
  Using cached numpy-2.5.4-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.1.1-cp313-cp313-macosx_11_0_arm64.whl (11.2 MB)
Using cached numpy-2.5.4-cp313-cp313-macosx_14_0_arm64.whl (5.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [5]:
g.search("Agra").columns


Index(['geonameid', 'name', 'asciiname', 'alternatenames', 'latitude',
       'longitude', 'feature_class', 'feature_code', 'country_code', 'cc2',
       'admin1_code', 'admin2_code', 'admin3_code', 'admin4_code',
       'population', 'elevation', 'dem', 'timezone', 'modification_date'],
      dtype='str')

In [7]:
g.search("kasol")


,geonameid,name,asciiname,alternatenames,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,population,elevation,dem,timezone,modification_date
14842,1267561,Kasol,Kasol,"Kasol,Kasol',kacol,kasaula,kasol,kasola,Касоль...",32.00988,77.31421,P,PPL,IN,,11,026,6887,,0,,1575,Asia/Kolkata,2024-01-06
182071,10514115,Kāsol,Kasol,,20.21460,77.29470,P,PPL,IN,,16,502,3996,,0,,526,Asia/Kolkata,2024-01-06
371359,10733229,Kasol,Kasol,,31.37082,76.86512,P,PPL,IN,,11,030,156,,0,,605,Asia/Kolkata,2024-01-06


In [ ]:
import pandas as pd

cities = pd.read_csv("raw_data/top_cities.csv")
cities["found"] = cities["city"].apply(lambda c: len(g.search(c)) > 0)

print(cities["found"].sum(), "/", len(cities), "found")
cities[~cities["found"]]

In [ ]:
from llm import LLM
LLM().ask("Lucknow se Jaipur jaana hai")


## Sarvam STT spike
Which model and mode give the agent the best input? `translit` = Roman Hinglish ("Lucknow se Jaipur jaana hai"), which matches what the agent is tested on. Run from the project root (the kernel's working directory).

In [8]:
# 1. Setup: one function that sends an audio file to Sarvam and returns the transcript
import json, os, time
from pathlib import Path

import httpx2 as httpx          # the httpx that the openai package already installed
import pandas as pd
from dotenv import load_dotenv

load_dotenv(".env")
SARVAM_URL = "https://api.sarvam.ai/speech-to-text"
SAMPLES = Path("raw_data/stt_samples")     # 16 kHz mono test clips (made with macOS `say`)


def transcribe(path, model="saaras:v4", mode=None, language_code="unknown", keyterms=None):
    """mode: transcribe (default) | codemix | translit (Roman Hinglish) | translate (English) | verbatim"""
    data = {"model": model, "language_code": language_code}
    if mode:
        data["mode"] = mode
    if keyterms:
        data["keyterms"] = json.dumps(keyterms)
    t0 = time.perf_counter()
    with open(path, "rb") as f:
        r = httpx.post(SARVAM_URL, headers={"api-subscription-key": os.environ["SARVAM_API_KEY"]},
                       data=data, files={"file": (Path(path).name, f)}, timeout=30)
    ms = round((time.perf_counter() - t0) * 1000)
    if r.status_code != 200:
        return {"transcript": None, "error": f"{r.status_code}: {r.text[:200]}", "ms": ms}
    j = r.json()
    return {"transcript": j["transcript"], "lang": j.get("language_code"), "ms": ms}

In [9]:
# 2. One clip through every model x mode, to pick the output format the agent wants
clip = SAMPLES / "01_lucknow_jaipur.wav"
pd.DataFrame([{"model": m, "mode": mode or "(default)", **transcribe(clip, m, mode)}
              for m in ["saaras:v4", "saaras:v3"]
              for mode in [None, "codemix", "translit", "translate"]])

,model,mode,transcript,lang,ms
0,saaras:v4,(default),"लखनऊ से जयपुर जाना है, बीच में आगरा।",hi-IN,597
1,saaras:v4,codemix,"लखनऊ से जयपुर जाना है, बीच में आगरा।",hi-IN,389
2,saaras:v4,translit,"Lucknow se Jaipur jaana hai, beech mein Agra.",hi-IN,460
3,saaras:v4,translate,"I have to go from Lucknow to Jaipur, with Agra...",hi-IN,556
4,saaras:v3,(default),"लखनऊ से जयपुर जाना है, बीच में आगरा।",hi-IN,420
5,saaras:v3,codemix,"लखनऊ से जयपुर जाना है, बीच में आगरा।",hi-IN,403
6,saaras:v3,translit,"Lucknow se Jaipur jaana hai, beech mein Agra,",hi-IN,370
7,saaras:v3,translate,"Have to go from Lucknow to Jaipur, Agra in bet...",hi-IN,421


In [10]:
# 3. All clips, the two candidate modes side by side. Add your own recordings to raw_data/stt_samples/
#    (any of wav/mp3/m4a/webm; record with QuickTime or your phone) - real voices matter more than `say`.
rows = []
for clip in sorted(SAMPLES.iterdir()):
    for mode in ["translit", "codemix"]:
        rows.append({"clip": clip.name, "mode": mode, **transcribe(clip, mode=mode)})
df = pd.DataFrame(rows)
print("median latency ms:", df["ms"].median())
df

median latency ms: 412.5


,clip,mode,transcript,lang,ms
0,01_lucknow_jaipur.wav,translit,"Lucknow se Jaipur jaana hai, beech mein Agra.",hi-IN,466
1,01_lucknow_jaipur.wav,codemix,"लखनऊ से जयपुर जाना है, बीच में आगरा।",hi-IN,437
2,02_agra_hatao.wav,translit,Agra hata do.,hi-IN,376
3,02_agra_hatao.wav,codemix,आगरा हटा दो।,hi-IN,334
4,03_udaipur_jodo.wav,translit,Jaipur ke baad Udaipur jodo.,hi-IN,382
5,03_udaipur_jodo.wav,codemix,जयपुर के बाद उदयपुर जोड़ो।,hi-IN,359
6,04_undo.wav,translit,Wapas pehle jaisa kar do.,hi-IN,373
7,04_undo.wav,codemix,वापस पहले जैसा कर दो।,hi-IN,435
8,05_kasol_manali.wav,translit,"Delhi se Kasol, phir Manali aur raaste mein Ch...",hi-IN,429
9,05_kasol_manali.wav,codemix,"दिल्ली से कसोल, फिर मनाली और रास्ते में चंडीगढ...",hi-IN,396


In [11]:
# 4. Keyterms (v4 only): bias recognition toward place names. Max 50 terms, so pass the trip's
#    current stops + a few big cities, not all 396 places.
terms = ["Kasol", "Manali", "Chandigarh", "Delhi", "Udaipur", "Kochi", "Coimbatore", "Mysore"]
clip = SAMPLES / "05_kasol_manali.wav"
pd.DataFrame([
    {"keyterms": "none", **transcribe(clip, mode="translit")},
    {"keyterms": "places", **transcribe(clip, mode="translit", keyterms=terms)},
])

,keyterms,transcript,lang,ms
0,none,"Delhi se Kasol, phir Manali aur raaste mein Ch...",hi-IN,585
1,places,"Delhi se Kasol, phir Manali aur raaste mein Ch...",hi-IN,538


In [ ]:
# 5. End to end: voice clip -> transcript -> agent (same as typing it in the text box)
import uuid
from app import agent, state

sid = uuid.uuid4().hex
for clip in ["01_lucknow_jaipur.wav", "03_udaipur_jodo.wav", "02_agra_hatao.wav", "04_undo.wav"]:
    heard = transcribe(SAMPLES / clip, mode="translit")["transcript"]
    reply = agent.run_turn(sid, heard)
    print(f"heard: {heard}\n  -> {state.stop_ids(sid)} | {reply}\n")